# 020 — Training: the four base architectures

Trains `unet`, `resunet`, `attention_unet`, `efficientnet_unet` on the RGB → IR translation task — the four deterministic (single-output) architectures that predate the `_nll`/`_v2`/`_restormer` variants. Companion notebooks, all sharing this same block layout for easy side-by-side comparison:

| notebook | trains | checkpoints |
|---|---|---|
| **020** (this one) | `unet`, `resunet`, `attention_unet`, `efficientnet_unet` | `models/deterministic/<arch>/` |
| `021_training_nll.ipynb` | the same four, with a heteroscedastic `(mu, log_var)` head, `gaussian_nll` loss | `models/nll_gaussian/<arch>_nll/` |
| `022_training_v2.ipynb` | the same four `_nll` architectures, `beta_nll` loss | `models/nll_beta/<arch>_nll/` |
| `023_training_variants.ipynb` | `unet_v2`, `unet_restormer` — architectural variants of `unet` | `models/deterministic/<arch>/` |

Only the **artwork-and-mockups** split is used (see §1) — it is the split every checkpoint in this project is actually trained and evaluated with.


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory (needed whether Jupyter was launched from the repo root or from `notebooks/`).


In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed (`scripts.reproducibility.set_global_seed`, reads `settings.SEED`) so this run is reproducible, and a GPU sanity check — training four architectures is expensive enough that silently falling back to CPU is worth catching immediately.


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import compile_model, get_callbacks, get_model
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Real artworks are grouped and kept entirely within one fold — no painting leaks across train/val/test. The mockup groups (`settings.MOCKUP_ARTWORK_IDS`) exist purely to be learned from, so a whole-group holdout would waste them; they are split at the individual-pair level instead, with only a small fraction (`settings.MOCKUP_TEST_RATIO`, default 5%) held out for test. This is the split every training/evaluation notebook in the project uses — see `notebooks/010_eda.ipynb` §2 for the split-integrity check and per-fold artwork breakdown.

`crop_size` only affects the augmented training split; validation stays full-image so metrics reflect real inference conditions.


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss functions

| Architecture | Loss |
|---|---|
| `unet`, `resunet`, `attention_unet` | `combined_loss` — MAE + (1 − SSIM) |
| `efficientnet_unet` | `combined_loss_advanced` — MAE + Laplacian pyramid + FFT |

`scripts.trainer.uses_advanced_loss` is the single source of truth for which architecture gets which loss — both training (below) and checkpoint loading (`030_evaluation.ipynb`) derive it from there, so the two paths can't drift apart. `efficientnet_unet`'s pretrained encoder reproduces high-frequency detail well, so it gets the loss that explicitly rewards it: the **Laplacian pyramid** term decomposes the error into spatial-frequency bands and weights finer detail more heavily; the **FFT** term penalises magnitude-spectrum error uniformly across all frequencies, preventing the model from trading away high-frequency accuracy for a lower low-frequency error. Both target the frequency band where underdrawing strokes live.

The cell below visualises the pyramid decomposition and FFT spectra of one training sample, to make those two terms concrete before training starts.


In [ ]:
import numpy as np
from PIL import Image

# --- Laplacian pyramid on a sample IR image ---
ir_np = np.array(Image.open(train_pairs[0][1]).convert("L")).astype(np.float32) / 255.0
ir_t = tf.constant(ir_np[np.newaxis, ..., np.newaxis])  # (1, H, W, 1)


def _lap_level(x: tf.Tensor) -> tuple:
    low = tf.nn.avg_pool2d(x, ksize=2, strides=2, padding="VALID")
    up = tf.image.resize(low, tf.shape(x)[1:3], method="bilinear")
    detail = (x - up)[0, ..., 0].numpy()
    return detail, low


LEVELS = 5
details, x = [], ir_t
for _ in range(LEVELS):
    d, x = _lap_level(x)
    details.append(d)

fig, axes = plt.subplots(1, LEVELS + 1, figsize=(4 * (LEVELS + 1), 4))
fig.suptitle("Laplacian pyramid — sample IR image  (level 0 = finest detail)")
axes[0].imshow(ir_np, cmap="gray")
axes[0].set_title("Original IR")
axes[0].axis("off")
for i, d in enumerate(details):
    d_disp = (d - d.min()) / (d.max() - d.min() + 1e-8)
    axes[i + 1].imshow(d_disp, cmap="RdBu_r")
    axes[i + 1].set_title(f"Level {i}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

# --- FFT magnitude spectra ---
rgb_np = (
    np.array(Image.open(train_pairs[0][0]).convert("RGB")).astype(np.float32) / 255.0
)
log_rgb = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(rgb_np[..., 0]))))
log_ir = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(ir_np))))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("FFT magnitude spectra (log scale) — same patch")
axes[0].imshow(log_rgb, cmap="inferno")
axes[0].set_title("RGB (channel R)")
axes[0].axis("off")
axes[1].imshow(log_ir, cmap="inferno")
axes[1].set_title("IR")
axes[1].axis("off")
axes[2].imshow(np.abs(log_rgb - log_ir), cmap="hot")
axes[2].set_title("Spectral difference |R − IR|")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 3. Train all four architectures

One loop over `ARCHS`, each compiled with its own loss (`compile_model` derives it from `uses_advanced_loss`) and trained with early stopping/`ReduceLROnPlateau` (`scripts.trainer.get_callbacks`). Checkpoints go to `models/deterministic/<arch>/best_model.keras`. Set `EPOCHS = 2` for a quick smoke test before committing to a full run — a real run takes several hours across all four architectures.


In [ ]:
ARCHS = ["unet", "resunet", "attention_unet", "efficientnet_unet"]
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
MODEL_DIR = settings.MODELS_DIR / "deterministic"
LOG_DIR = settings.LOGS_DIR / "deterministic"

histories: dict = {}

for arch in ARCHS:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}")
    print(f"{'=' * 60}")

    model = get_model(arch)
    model = compile_model(
        model, arch, lr=settings.LEARNING_RATE, loss_alpha=settings.LOSS_ALPHA
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(arch, log_dir=LOG_DIR, model_dir=MODEL_DIR)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}): {best_val_loss:.4f}")

## 4. Training curves


In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

## 5. Summary

Checkpoints saved to `models/deterministic/<arch>/best_model.keras`. Logs written to `logs/deterministic/<arch>/` — run `tensorboard --logdir logs/` to inspect curves interactively.

> **Note:** `efficientnet_unet` downloads EfficientNetB0 ImageNet weights on first use. Subsequent runs use the local Keras cache.


In [ ]:
for arch in ARCHS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")